<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/eval_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Small VLMs Evaluation Pipeline with G-Eval

## Initial steps

In [1]:
!pip install --q supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.4 MB/s eta 0:00:00


In [2]:
import re
import math
import numpy as np
import pandas as pd
from google.colab import userdata
from supabase import create_client

In [3]:
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [12]:
# Load Supabase for analysis
all_rows  = []
page_size = 1000
offset    = 0

while True:
    resp = supabase.table("eval_scores").select(
        "model_name, metadata_id, dashboard_name, chart_id, chart_title, level, "
        "generated, reference, generation_success, degenerate_pattern, "
        "rouge1, rouge2, rougel, bertscore_f1, "
        "geval, geval_reason"
    ).range(offset, offset + page_size - 1).execute()
    batch = resp.data
    if not batch:
        break
    all_rows.extend(batch)
    if len(batch) < page_size:
        break
    offset += page_size

df_qual = pd.DataFrame(all_rows)
df_qual = df_qual.rename(columns={"rougel": "rougeL"})
df_qual = df_qual[df_qual["model_name"] != "SmolVLM-256M-Instruct"].copy()

print(f"df_qual shape : {df_qual.shape}")
print(f"Models        : {sorted(df_qual['model_name'].unique())}")
print(f"Columns       : {df_qual.columns.tolist()}")

df_qual shape : (6297, 16)
Models        : ['InternVL2-1B', 'InternVL2-2B', 'InternVL3-1B-hf', 'Qwen3-VL-2B-Instruct', 'Qwen3.5-0.8B', 'Qwen3.5-2B', 'moondream2']
Columns       : ['model_name', 'metadata_id', 'dashboard_name', 'chart_id', 'chart_title', 'level', 'generated', 'reference', 'generation_success', 'degenerate_pattern', 'rouge1', 'rouge2', 'rougeL', 'bertscore_f1', 'geval', 'geval_reason']


In [13]:
df_qual.describe()

,chart_id,rouge1,rouge2,rougeL,bertscore_f1,geval
count,6297.000000,2081.000000,2081.000000,2081.000000,2081.000000,2012.000000
mean,9.045260,0.214284,0.057583,0.179507,0.211118,0.221582
std,13.750962,0.152036,0.090972,0.129745,0.179794,0.213596
min,1.000000,0.000000,0.000000,0.000000,-0.403514,0.000000
25%,2.000000,0.102564,0.000000,0.093023,0.100174,0.004500
50%,5.000000,0.200000,0.000000,0.166667,0.220669,0.212250
75%,10.000000,0.297872,0.090909,0.242424,0.330066,0.355600
max,106.000000,0.827586,0.628571,0.827586,0.800141,1.000000


In [14]:
df_qual.head()

,model_name,metadata_id,dashboard_name,chart_id,chart_title,level,generated,reference,generation_success,degenerate_pattern,rouge1,rouge2,rougeL,bertscore_f1,geval,geval_reason
0,Qwen3-VL-2B-Instruct,abee2e83-6384-4c23-abfd-e5ede8b5a7bf,Superstore Sales Overview,1,Scoreboard Overview,L2,"Sales $733.2K, Profit $93.4K, Orders 1,687, Qu...","Sales are about $733.2K, profit is $93.4K, ord...",True,None,0.774194,0.413793,0.774194,0.720033,NaN,None
1,Qwen3-VL-2B-Instruct,abee2e83-6384-4c23-abfd-e5ede8b5a7bf,Superstore Sales Overview,2,Sales Target,L2,"Total Sales $733.2K, Target Sales $1.0M, Perce...","With total sales of $733.2K (73.3%), which is ...",True,None,0.645161,0.275862,0.451613,0.473081,NaN,None
2,Qwen3-VL-2B-Instruct,abee2e83-6384-4c23-abfd-e5ede8b5a7bf,Superstore Sales Overview,2,Sales Target,L3,"The target is 73.3% of the actual sales, indic...",None,True,None,NaN,NaN,NaN,NaN,NaN,None
3,Qwen3-VL-2B-Instruct,abee2e83-6384-4c23-abfd-e5ede8b5a7bf,Superstore Sales Overview,3,Monthly Target,L2,"The monthly target is $100K, with the highest ...","Out of the monthly sales, only in November cou...",True,None,0.444444,0.080000,0.296296,0.440662,NaN,None
4,Qwen3-VL-2B-Instruct,abee2e83-6384-4c23-abfd-e5ede8b5a7bf,Superstore Sales Overview,4,Sales by Sub-Category,L2,"Phones 33%, Chairs 14%, Binders 47%, Storage 1...","California leads the sales by $146,4K, while N...",True,None,0.000000,0.000000,0.000000,-0.248635,NaN,None


## Evaluation results

### Performance across semantic levels

In [15]:
def level_summary(df, metric_col, levels):
    sub = df[df["level"].isin(levels) & df[metric_col].notna()]
    return sub.groupby("level")[metric_col].agg(
        mean="mean", sd="std"
    ).round(4)

def fmt_mean_sd(df_summary, label):
    col = df_summary.apply(
        lambda r: f"{r['mean']:.3f} ± {r['sd']:.3f}" if pd.notna(r['mean']) else "-",
        axis=1
    ).rename(label)
    return col

rouge1_summary = level_summary(df_qual, "rouge1",       ["L2","L3"])
rouge2_summary = level_summary(df_qual, "rouge2",       ["L2","L3"])
rougeL_summary = level_summary(df_qual, "rougeL",       ["L2","L3"])
bert_summary   = level_summary(df_qual, "bertscore_f1", ["L2","L3"])
geval_summary  = level_summary(df_qual, "geval",        ["L3","L4"])

table_43 = pd.concat([
    fmt_mean_sd(rouge1_summary, "ROUGE-1"),
    fmt_mean_sd(rouge2_summary, "ROUGE-2"),
    fmt_mean_sd(rougeL_summary, "ROUGE-L"),
    fmt_mean_sd(bert_summary,   "BERTScore"),
    fmt_mean_sd(geval_summary,  "G-Eval"),
], axis=1).reindex(["L2","L3","L4"]).fillna("-")

print("=== Performance Across Semantic Levels (mean ± SD) ===")
print(table_43.to_string())

=== Performance Across Semantic Levels (mean ± SD) ===
             ROUGE-1        ROUGE-2        ROUGE-L      BERTScore         G-Eval
level                                                                           
L2     0.228 ± 0.176  0.078 ± 0.107  0.195 ± 0.154  0.213 ± 0.206              -
L3     0.198 ± 0.113  0.032 ± 0.057  0.160 ± 0.089  0.208 ± 0.141  0.143 ± 0.223
L4                 -              -              -              -  0.290 ± 0.179


### Performance across models


In [16]:
def fmt(mean, sd):
    if pd.isna(mean):
        return "-"
    return f"{mean:.3f} ± {sd:.3f}"

rows = []
for model in sorted(df_qual["model_name"].unique()):
    m = df_qual[df_qual["model_name"] == model]

    def ms(level, metric):
        sub = m[(m["level"] == level) & m[metric].notna()][metric]
        return sub.mean(), sub.std()

    for level in ["L2", "L3", "L4"]:
        row = {
            "Model"    : model if level == "L2" else "",
            "Level"    : level,
            "ROUGE-1"  : fmt(*ms(level, "rouge1"))      if level in ["L2","L3"] else "-",
            "ROUGE-2"  : fmt(*ms(level, "rouge2"))      if level in ["L2","L3"] else "-",
            "ROUGE-L" : fmt(*ms(level, "rougeL"))      if level in ["L2","L3"] else "-",
            "BERTScore": fmt(*ms(level, "bertscore_f1"))if level in ["L2","L3"] else "-",
            "G-Eval"   : fmt(*ms(level, "geval"))       if level in ["L3","L4"] else "-",
        }
        rows.append(row)

table_44 = pd.DataFrame(rows).set_index(["Model","Level"])
print("\n=== Model x Metric Performance (mean ± SD) ===")
print(table_44.to_string())


=== Model x Metric Performance (mean ± SD) ===
                                  ROUGE-1        ROUGE-2        ROUGE-L      BERTScore         G-Eval
Model                Level                                                                           
InternVL2-1B         L2     0.131 ± 0.131  0.038 ± 0.073  0.127 ± 0.126  0.113 ± 0.190              -
                     L3     0.082 ± 0.084  0.006 ± 0.027  0.075 ± 0.071  0.065 ± 0.137  0.051 ± 0.122
                     L4                 -              -              -              -  0.118 ± 0.123
InternVL2-2B         L2     0.205 ± 0.142  0.056 ± 0.082  0.172 ± 0.118  0.193 ± 0.184              -
                     L3     0.161 ± 0.105  0.019 ± 0.039  0.136 ± 0.089  0.191 ± 0.136  0.075 ± 0.131
                     L4                 -              -              -              -  0.242 ± 0.137
InternVL3-1B-hf      L2     0.224 ± 0.112  0.064 ± 0.070  0.193 ± 0.095  0.219 ± 0.164              -
                     L3     0.213 

### Performance within family

In [17]:
FAMILY_PAIRS = {
    "Qwen3.5"  : ("Qwen3.5-0.8B", "Qwen3.5-2B"),
    "InternVL2": ("InternVL2-1B",  "InternVL2-2B"),
}

METRICS_PER_LEVEL = {
    "rougeL"      : ["L2", "L3"],
    "bertscore_f1": ["L2", "L3"],
    "geval"       : ["L3", "L4"],
}

scaling_rows = []
for family, (small, large) in FAMILY_PAIRS.items():
    first_row = True
    for metric, levels in METRICS_PER_LEVEL.items():
        first_metric = True
        for level in levels:
            sub        = df_qual[df_qual["level"] == level]
            small_mean = sub[sub["model_name"] == small][metric].mean()
            large_mean = sub[sub["model_name"] == large][metric].mean()
            delta      = round(large_mean - small_mean, 4) if not (
                pd.isna(small_mean) or pd.isna(large_mean)
            ) else None

            scaling_rows.append({
                "Family": family if first_row else "",
                "Metric": metric if first_metric else "",
                "Level" : level,
                "Small" : round(small_mean, 4) if not pd.isna(small_mean) else "-",
                "Large" : round(large_mean, 4) if not pd.isna(large_mean) else "-",
                "Delta" : delta if delta is not None else "-",
            })
            first_row    = False
            first_metric = False

df_scaling = pd.DataFrame(scaling_rows)
print("\n=== Within-Family Parameter Scaling ===")
print(df_scaling.to_string(index=False))


=== Within-Family Parameter Scaling ===
   Family       Metric Level  Small  Large  Delta
  Qwen3.5       rougeL    L2 0.1247 0.2996 0.1749
                          L3 0.1867 0.1942 0.0075
          bertscore_f1    L2 0.1588 0.3471 0.1884
                          L3 0.2316 0.2741 0.0425
                 geval    L3 0.1022 0.2750 0.1728
                          L4 0.2742 0.4245 0.1504
InternVL2       rougeL    L2 0.1270 0.1718 0.0447
                          L3 0.0752 0.1359 0.0608
          bertscore_f1    L2 0.1133 0.1930 0.0797
                          L3 0.0649 0.1906 0.1257
                 geval    L3 0.0511 0.0749 0.0238
                          L4 0.1179 0.2417 0.1239


## Qualitative analysis

### Hardest dashboards

In [18]:
# Per dashboard: average score across all models and all levels.
# Composite uses rougeL, bertscore_f1, and geval.

def row_composite(r):
    vals = []
    if pd.notna(r.get("rougeL")):       vals.append(r["rougeL"])
    if pd.notna(r.get("bertscore_f1")): vals.append(r["bertscore_f1"])
    if pd.notna(r.get("geval")):   vals.append(r["geval"])
    return np.mean(vals) if vals else np.nan

df_qual["composite"] = df_qual.apply(row_composite, axis=1)

dashboard_difficulty = (
     df_qual.groupby(["metadata_id","dashboard_name"])["composite"]
    .mean()
    .reset_index()
    .rename(columns={"composite": "avg_composite_score"})
    .sort_values("avg_composite_score")
)

meta_resp = supabase.table("metadata").select("id, dashboard_author").execute()
df_meta_full = pd.DataFrame(meta_resp.data).rename(columns={"id": "metadata_id"})

dashboard_difficulty = dashboard_difficulty.merge(
    df_meta_full[["metadata_id","dashboard_author"]],
    on="metadata_id",
    how="left"
)

N_HARD = 5
display_cols = ["dashboard_name","dashboard_author","avg_composite_score"]

print(f"=== Hardest Dashboards ===")
print(dashboard_difficulty[display_cols].head(N_HARD).round(4).to_string(index=False))
print(f"\n=== Easiest Dashboards ===")
print(dashboard_difficulty[display_cols].tail(N_HARD).round(4).to_string(index=False))

=== Hardest Dashboards ===
                                        dashboard_name dashboard_author  avg_composite_score
                       Superstore Performance Overview         Julie Li               0.1397
                         Superstore Dashboard Overview        Cathy Lau               0.1433
              Superstore Dashboard - Business Overview    Murilo Cremon               0.1606
                             Superstore Sales Overview     Keren Aharon               0.1681
Superstore Order Details | KPI's and Selection Filters   Naresh Suglani               0.1711

=== Easiest Dashboards ===
                                   dashboard_name dashboard_author  avg_composite_score
                                  Superstore KPIs     Andy Kriebel               0.2667
                 Superstore Sales Dashboard #VOTD    Israel Ayoola               0.2767
                                       Superstore        Bill Yost               0.2784
SuperStore Dashboard - 2019 Sales &

### Top scoring samples

In [19]:
TOP_N = 1

# Load author lookup once
meta_resp = supabase.table("metadata").select("id, dashboard_author").execute()
df_meta_full = pd.DataFrame(meta_resp.data).rename(columns={"id": "metadata_id"})

top_examples = []
for model in sorted(df_qual["model_name"].unique()):
    for level in ["L2", "L3", "L4"]:
        if level == "L4":
            metric = "geval"
        elif level == "L2":
            metric = "bertscore_f1"
        else:  # L3
            metric = "bertscore_f1"
        sub = df_qual[
            (df_qual["model_name"] == model) &
            (df_qual["level"]      == level) &
            df_qual[metric].notna()
        ].nlargest(TOP_N, metric)

        for _, r in sub.iterrows():
            top_examples.append({
                "model"            : model,
                "level"            : level,
                "dashboard_name"   : r["dashboard_name"],
                "metadata_id"      : r["metadata_id"],
                "chart_id"         : r["chart_id"],
                "score"            : round(r[metric], 4),
                "metric_used"      : metric,
                "generated"        : r["generated"],
                "reference"        : r["reference"],
            })

df_top = pd.DataFrame(top_examples).merge(
    df_meta_full[["metadata_id","dashboard_author"]],
    on="metadata_id",
    how="left"
)

print(f"Top examples collected: {len(df_top)}")
print(df_top[["model","level","dashboard_name","dashboard_author","chart_id","score","metric_used"]].head(21).to_string(index=False))

Top examples collected: 21
               model level                                     dashboard_name       dashboard_author  chart_id  score  metric_used
        InternVL2-1B    L2                               Superstore Dashboard          Harshit Gupta         3 0.5397 bertscore_f1
        InternVL2-1B    L3  SuperStore Dashboard - 2019 Sales & Profitability              Linh Pham         2 0.4519 bertscore_f1
        InternVL2-1B    L4                               Superstore Dashboard     Divas Pratap Singh         4 0.6976        geval
        InternVL2-2B    L2                          Superstore Sales Overview           Keren Aharon         1 0.6540 bertscore_f1
        InternVL2-2B    L3 Demo Excecutive Performance Dashboard - Superstore                An Tran         1 0.4696 bertscore_f1
        InternVL2-2B    L4                   Superstore Performance Dashboard         Tanya Lomskaya         5 0.7514        geval
     InternVL3-1B-hf    L2                              

### Low scoring samples

In [20]:
# Low scoring examples — L3 and L4 only, 1 per model per level
BOTTOM_N = 1

meta_resp = supabase.table("metadata").select("id, dashboard_author").execute()
df_meta_full = pd.DataFrame(meta_resp.data).rename(columns={"id": "metadata_id"})

bottom_examples = []

for model in sorted(df_qual["model_name"].unique()):
    for level in ["L3", "L4"]:
        metric = "geval"

        sub = df_qual[
            (df_qual["model_name"] == model) &
            (df_qual["level"]      == level) &
            df_qual[metric].notna()
        ]

        for _, r in sub.nsmallest(BOTTOM_N, metric).iterrows():
            bottom_examples.append({
                "model"         : model,
                "level"         : level,
                "dashboard_name": r["dashboard_name"],
                "metadata_id"   : r["metadata_id"],
                "chart_id"      : r["chart_id"],
                "score"         : round(r[metric], 4),
                "generated"     : r["generated"],
                "reference"     : r["reference"],
                "geval_reason"  : r.get("geval_reason", None),
            })

df_bottom = pd.DataFrame(bottom_examples).merge(
    df_meta_full[["metadata_id","dashboard_author"]], on="metadata_id", how="left"
)

print(f"=== Bottom scoring examples per model for L3 and L4 ===")
print(df_bottom[["model","level","dashboard_name","dashboard_author",
                 "chart_id","score","geval_reason"]].to_string(index=False))

=== Bottom scoring examples per model for L3 and L4 ===
               model level                              dashboard_name  dashboard_author  chart_id  score                                                                                                                                                                                                                                                                                                                                                                                  geval_reason
        InternVL2-1B    L3  Interactive Drill Down Superstore Overview      Juliet Craig         1 0.0000                                                                           The actual output identifies a ranking of states by sales revenue, which is a cross-segment synthesis, while the expected output describes a trend over time, specifically higher sales at the end of the year. The pattern types do not match, and the actual output does not addre

### Failure mode taxonomy

In [21]:
# Classify each VLM output into a failure mode
# Applied to all rows regardless of level

def classify_failure(row):
    gen = row.get("generated")
    if gen is None or (isinstance(gen, float) and math.isnan(gen)):
        return "unparseable_or_empty"
    gen_lower   = str(gen).lower()
    ref         = row.get("reference")
    ref_is_null = ref is None or (isinstance(ref, float) and math.isnan(ref))
    GENERIC_L4 = [
        "it is important to", "organizations should", "decision makers",
        "stakeholders", "further investigation", "may indicate",
    ]
    if row["level"] == "L4" and any(s in gen_lower for s in GENERIC_L4):
        return "generic_L4"
    if not ref_is_null:
        gen_nums = set(re.findall(r"\b\d[\d,.]*\b", str(gen)))
        ref_nums = set(re.findall(r"\b\d[\d,.]*\b", str(ref)))
        if len(gen_nums - ref_nums) >= 3:
            return "possible_hallucination"
    return "no_issue_detected"

df_qual["failure_mode"] = df_qual.apply(classify_failure, axis=1)

failure_summary = (
    df_qual.groupby(["model_name", "failure_mode"])
    .size()
    .reset_index(name="count")
    .pivot(index="model_name", columns="failure_mode", values="count")
    .fillna(0).astype(int)
)

failure_summary["total"] = failure_summary.sum(axis=1)

print("=== Failure Mode Taxonomy ===")
print(failure_summary.to_string())

=== Failure Mode Taxonomy ===
failure_mode          generic_L4  no_issue_detected  possible_hallucination  unparseable_or_empty  total
model_name                                                                                              
InternVL2-1B                   0               1080                      18                   768   1866
InternVL2-2B                   0                671                      45                     4    720
InternVL3-1B-hf                1                927                      83                    48   1059
Qwen3-VL-2B-Instruct          18                502                      81                     2    603
Qwen3.5-0.8B                   8               1008                      29                    14   1059
Qwen3.5-2B                     0                569                      49                     0    618
moondream2                     0                335                      11                    26    372


### Side-by-side qualitative comparison

In [22]:
def show_example(model, dashboard_name, chart_id, level):
    row = df_qual[
        (df_qual["model_name"]     == model) &
        (df_qual["dashboard_name"] == dashboard_name) &
        (df_qual["chart_id"]       == chart_id) &
        (df_qual["level"]          == level)
    ]
    if row.empty:
        print("No matching row found.")
        return
    r = row.iloc[0]
    print(f"Model      : {r['model_name']}")
    print(f"Dashboard  : {r['dashboard_name']}")
    print(f"Chart      : {r['chart_id']} - {r.get('chart_title','')}")
    print(f"Level      : {r['level']}")
    print(f"\nGenerated  :\n  {r['generated']}")
    print(f"\nReference  :\n  {r['reference']}")
    print(f"\nROUGE-1    : {round(r['rouge1'], 4) if pd.notna(r.get('rouge1')) else '-'}")
    print(f"ROUGE-2    : {round(r['rouge2'], 4) if pd.notna(r.get('rouge2')) else '-'}")
    print(f"ROUGE-L    : {round(r['rougeL'], 4) if pd.notna(r.get('rougeL')) else '-'}")
    print(f"BERTScore  : {round(r['bertscore_f1'], 4) if pd.notna(r.get('bertscore_f1')) else '-'}")
    print(f"G-Eval     : {r.get('geval','-')}")
    print(f"Failure    : {r.get('failure_mode','-')}")
    print("-" * 70)

# Example call with any row from df_top
show_example(
    model          = df_top.iloc[0]["model"],
    dashboard_name = df_top.iloc[0]["dashboard_name"],
    chart_id       = df_top.iloc[0]["chart_id"],
    level          = df_top.iloc[0]["level"],
)

Model      : InternVL2-1B
Dashboard  : Superstore Dashboard
Chart      : 3 - Sales by Sub-Category
Level      : L2

Generated  :
  Accessories: 7.2%, Art: 1.2%, Binders: 1.2%, Bookcases: 8.9%, Chairs: 5.0%, Cops: 4.4%, Envelopes: 6.5%, Fasteners: 0.7%, Furnishings: 0.4%, Labels: 4.1%, Machines: 0.5%, Paper: 3.4%, Phones: 14.3%, Storage: 9.7%, Supplies: 2.0%, Tables: 8.9%

Reference  :
  Chairs have the highest sales by sub-category at $336K, followed by phones at $332K, and the lowest sales are in Fasteners at $9K

ROUGE-1    : 0.0833
ROUGE-2    : 0.0
ROUGE-L    : 0.0556
BERTScore  : -0.0982
G-Eval     : nan
Failure    : possible_hallucination
----------------------------------------------------------------------
